# 01 — Ground Tabular Data: Initial EDA and Schema Standardization


## Purpose
This notebook performs a first-pass exploratory analysis of the ground (tabular) datasets for both sites.
It focuses on:
- Loading and validating raw files
- Identifying timestamp and target (GHI) columns
- Verifying time frequency and missingness patterns
- Producing a standardized, site-agnostic view of the data schema

## Imports and settings

In [1]:
from __future__ import annotations

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path("..").resolve()
DATA_RAW = PROJECT_ROOT / "data_raw"

UNI_PATH = DATA_RAW / "Data_Uniandes_tierra.csv"  
ELP_PATH = DATA_RAW / "Data_ElPaso_tierra.csv"

TIMEZONE = "America/Bogota"
FREQ = "10min"   # expected sampling frequency for this project

print("PROJECT_ROOT:", PROJECT_ROOT)
print("UNI_PATH exists:", UNI_PATH.exists(), "->", UNI_PATH)
print("ELP_PATH exists:", ELP_PATH.exists(), "->", ELP_PATH)


PROJECT_ROOT: /srv/projects/Proyecto_e_ladino
UNI_PATH exists: True -> /srv/projects/Proyecto_e_ladino/data_raw/Data_Uniandes_tierra.csv
ELP_PATH exists: True -> /srv/projects/Proyecto_e_ladino/data_raw/Data_ElPaso_tierra.csv


## Uniandes

In [2]:
uni_df = pd.read_csv(UNI_PATH)

print("UNI shape:", uni_df.shape)
print("\nUNI columns:")
print(list(uni_df.columns))

print("\nUNI head:")
display(uni_df.head())


UNI shape: (165312, 1)

UNI columns:
['Timestamp;"Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Humedad relativa del aire 1 [%] - E_AH_REL1";"Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Presión relativa del aire 1 [hPa] - E_AP_REL1";"Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Dirección del viento [°] - E_W_D";"Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Radiación [W/m²] - SRAD";"Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Temperatura [°C] - T";"Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Velocidad del viento [m/s] - E_W_S"']

UNI head:


,,,,,,"Timestamp;""Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Humedad relativa del aire 1 [%] - E_AH_REL1"";""Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Presión relativa del aire 1 [hPa] - E_AP_REL1"";""Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Dirección del viento [°] - E_W_D"";""Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Radiación [W/m²] - SRAD"";""Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Temperatura [°C] - T"";""Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Velocidad del viento [m/s] - E_W_S"""
2023-09-01 00:00:00;73,896;745,288;163,000;0,000;12,718;1,42.0
2023-09-01 00:05:00;74,888;745,184;108,532;0,000;12,292;2,858.0
2023-09-01 00:10:00;76,484;745,140;113,694;0,000;11,932;2,728.0
2023-09-01 00:15:00;76,674;745,032;110,576;0,144;12,132;2,300.0
2023-09-01 00:20:00;75,824;744,972;116,006;0,000;12,318;1,346.0


## El Paso

In [3]:
elp_df = pd.read_csv(ELP_PATH)

print("ELP shape:", elp_df.shape)
print("\nELP columns:")
print(list(elp_df.columns))

print("\nELP head:")
display(elp_df.head())


ELP shape: (107172, 10)

ELP columns:
['Unnamed: 0', 'CSI', 'GHI', 'Presion', 'TempAmb', 'Wind Y', 'Wind X', 'DoY Sin', 'DoY Cos', 'horas']

ELP head:


,Unnamed: 0,CSI,GHI,Presion,TempAmb,Wind Y,Wind X,DoY Sin,DoY Cos,horas
0,2022-02-21 18:00:00+00:00,2.0,3.0352,1000.7912,29.9672,2.832954,-0.093612,0.778764,0.627317,18
1,2022-02-21 18:10:00+00:00,0.0,0.3562,1000.9321,29.5689,3.387552,0.796801,0.778764,0.627317,18
2,2022-02-21 18:20:00+00:00,0.0,0.0000,1001.1479,29.2593,2.091197,-0.878680,0.778764,0.627317,18
3,2022-02-21 18:30:00+00:00,0.0,0.0000,1001.2992,28.9183,-0.487957,-1.478562,0.778764,0.627317,18
4,2022-02-21 18:40:00+00:00,0.0,0.0000,1001.4676,28.5578,0.891171,-2.047462,0.778764,0.627317,18


## Sanity check

In [4]:
print("UNI dtypes:")
display(uni_df.dtypes)

print("\nELP dtypes:")
display(elp_df.dtypes)

UNI dtypes:


Timestamp;"Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Humedad relativa del aire 1 [%] - E_AH_REL1";"Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Presión relativa del aire 1 [hPa] - E_AP_REL1";"Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Dirección del viento [°] - E_W_D";"Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Radiación [W/m²] - SRAD";"Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Temperatura [°C] - T";"Sistema FV Andes 80kWp - Lufft WS501 BM: RS485-2 67 - Velocidad del viento [m/s] - E_W_S"    float64
dtype: object


ELP dtypes:


Unnamed: 0        str
CSI           float64
GHI           float64
Presion       float64
TempAmb       float64
Wind Y        float64
Wind X        float64
DoY Sin       float64
DoY Cos       float64
horas           int64
dtype: object